In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import os
import yaml
from pathlib import Path
from dask.distributed import Client
import dask.dataframe as dd
import networkx as nx
import sys
import pandas as pd


import ipycytoscape
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import intervals
import pygtrie
import seaborn as sns


/usr/workspace/pandey2/DFT/envDFT/lib/python3.9/site-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [3]:

use_local=False
if not use_local:
    with open(f'/g/g91/pandey2/.dftracer/configuration.yaml', 'r') as file:
        dlp_yaml = yaml.safe_load(file)
        app_root = dlp_yaml["app"]
else:
    app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)

import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer,get_dft_configuration,update_dft_configuration,setup_logging,setup_dask_cluster, reset_dask_cluster, get_dft_configuration
from dfanalyzer.graph_visualization.cytoscape import GraphFunctions, CytoGraph
from dfanalyzer.graph import DFGrepInterference, DFGrepWorkflow 

if not use_local:
    dask_run_dir = os.path.join(app_root, "dfanalyzer", "dask", "run_dir")
    with open (os.path.join(dask_run_dir, f"scheduler_{os.getenv('USER')}.json"), "r") as f:
        dask_scheduler = json.load(f)["address"]
else:
    dask_scheduler = None

# App Name
app_name = "deepspeedlow" 

condition_fn = None #

if app_name == "mummi":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/mummi-32-node/*pfw.gz"

elif app_name == "montage":
    filename ="/usr/workspace/iopp/graph-io/dlp_logs/montage_16_48ppn/montage*.pfw"
    cp_dir = "/p/lustre2/pandey2/cp_dir/montage"

elif app_name == "montage2m2d":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/montage-2mass-2-degree/*.pfw.gz"
    cp_dir = "/p/lustre2/pandey2/cp_dir/montage"

elif app_name == "montage2m7d":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/montage-2mass-7-degree/*.pfw.gz"
    cp_dir = "/p/lustre2/pandey2/cp_dir/montage"

elif app_name == "deepspeedlow":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/deepspeed_2_node_low_interference/*.pfw.gz"
    cp_dir = "/p/lustre2/pandey2/cp_dir/deepspeed"
elif app_name == "deepspeedhigh":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/deepspeed_2_node_high_interference/*.pfw.gz"
    cp_dir = "/p/lustre2/pandey2/cp_dir/deepspeed"
else:
    raise Exception("Unknown App name")


# Configuration 4 update log file dlp -> df
conf = update_dft_configuration(dask_scheduler=dask_scheduler, verbose=True, 
                                log_file=f"./dft_{os.getenv('USER')}.log", rebuild_index=False, time_approximate=False, 
                                host_pattern=r'lassen(\d+)', time_granularity=30e6, skip_hostname=True, conditions=condition_fn)
conf = get_dft_configuration()


# Setup
setup_logging()
setup_dask_cluster()
reset_dask_cluster()

[INFO] [06:06:47] Initialized Client with 768 workers and link http://134.9.71.28:8787/status [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:673]


/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/__init__.py


[INFO] [06:07:30] Restarting all workers [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:665]


In [5]:
def find_mount_point(path,trie):
    mount_point = trie.longest_prefix(path)
    if mount_point:
        return mount_point.key
    return '/'.join(path.split('/', 3)[:3])

def all_mount_points():
    with open("/proc/mounts", "r") as file:
        mount_points = [line.split()[1] for line in file]
    with open("/usr/workspace/pandey2/lassen_mounts", "r") as file:
        mount_p = [line.split()[1] for line in file]
    return mount_points+mount_p

mount_points = all_mount_points()
trie = pygtrie.StringTrie(zip(mount_points, [True] * len(mount_points)))


In [6]:
def deepspeed_cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return '/'.join(path.split('/', 3)[:3])
    
    if "args" in json_object:
        if "fname" in json_object["args"]:
            d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=str(json_object["args"]["fname"]))

    return d
load_cols_deepspeed = {'mount_point':"string[pyarrow]"}


In [7]:
analyzer_deepspeed = DFAnalyzer(filename,load_fn=deepspeed_cols_function, load_cols=load_cols_deepspeed, load_data={"mount_point":trie})

[INFO] [06:07:51] Created index for 16 files [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:371]
[INFO] [06:07:51] Total size of all files are <dask.bag.core.Item object at 0x1554ac13dbe0> bytes [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:373]
[INFO] [06:07:53] Loading 8992 batches out of 16 files and has 147114386 lines overall [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:386]
[INFO] [06:09:31] Loaded events [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:431]
[INFO] [06:09:31] Loaded plots with slope threshold: 45 [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:437]


In [8]:
analyzer_deepspeed.events['id'] = analyzer_deepspeed.events.index

In [14]:

# eventsDF = analyzer_montage.events[analyzer_montage.events['cat'] == "POSIX" ] # only posix events

In [9]:
filename

'/usr/workspace/iopp/graph-io/dlp_logs/deepspeed_2_node_high_interference/*.pfw.gz'

In [9]:
IFCalculator = DFGrepInterference(analyzer_deepspeed.events, app_name=app_name, cp_dir=cp_dir, existing=False)

In [10]:
IFCalculator.get_degree()
IFCalculator.get_interference()

In [ ]:
IFCalculator.write_checkpoint("inter", cp_dir=cp_dir)

In [10]:
IFCalculator.get_interference_metadata()

In [13]:
IFCalculator.write_checkpoint("inter", cp_dir=cp_dir)

In [11]:
IFCalculator.write_checkpoint("inter", cp_dir=cp_dir)
IFCalculator.write_checkpoint("inter_metadata", cp_dir=cp_dir)

In [13]:
IFCalculator.inter.compute()

,name,pid,size,ts,te,mount_point,dur,trange,deg_caller,deg_other,min_dur,interference
0,write,0,134217728,660590087,660950423,/p/lustre2,360336,22,1,1,335032,0.929777
1,write,0,134217728,661006110,661369818,/p/lustre2,363708,22,1,1,335032,0.921157
2,write,0,134217728,661438391,661805701,/p/lustre2,367310,22,3,1,335032,0.912123
3,write,12,134217728,661725733,662300105,/p/lustre2,574372,22,9,1,335032,0.583301
4,write,15,134217728,661787472,662355067,/p/lustre2,567595,22,10,1,335032,0.590266
...,...,...,...,...,...,...,...,...,...,...,...,...
580,write,0,134217728,1016927240,1017271769,/p/lustre2,344529,33,1,1,335032,0.972435
581,write,0,134217728,1017327292,1017767067,/p/lustre2,439775,33,1,1,335032,0.761826
582,write,0,134217728,1017850220,1018384076,/p/lustre2,533856,33,1,1,335032,0.62757
583,write,0,134217728,1018467788,1018989776,/p/lustre2,521988,33,1,1,335032,0.641839


In [14]:
x = IFCalculator.inter.compute()

In [16]:
x.groupby('mount_point').count()

,name,pid,size,ts,te,dur,trange,deg_caller,deg_other,min_dur,interference
mount_point,,,,,,,,,,,
/,8900,8900,8900,8900,8900,8900,8900,8900,8900,8900,8900
/p/lustre2,4696,4696,4696,4696,4696,4696,4696,4696,4696,4696,4696
